cve v5 

In [ ]:
from pathlib import Path
import shutil
import subprocess


def update_cvelist():
    historical_dir = Path.cwd()
    repository_dir = historical_dir / "cvelistV5-live"
    repository_url = "https://github.com/CVEProject/cvelistV5.git"

    git = shutil.which("git")

    if git is None:
        raise RuntimeError(
            "Git is not installed or is not on PATH. "
            "Install Git, then restart Jupyter."
        )

    def run_git(*arguments):
        result = subprocess.run(
            [git, *arguments],
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            raise RuntimeError(
                result.stderr.strip()
                or result.stdout.strip()
                or "Git command failed."
            )

        return result.stdout.strip()

    if not repository_dir.exists():
        print("First run: downloading the repository once...")

        run_git(
            "clone",
            "--depth", "1",
            "--single-branch",
            "--branch", "main",
            "--no-tags",
            repository_url,
            str(repository_dir),
        )

    else:
        # Never try to overwrite an ordinary extracted ZIP folder.
        if not (repository_dir / ".git").exists():
            raise RuntimeError(
                f"{repository_dir} exists but is not a Git clone. "
                "Use a different folder name or move that folder elsewhere."
            )

        remote = run_git(
            "-C", str(repository_dir),
            "remote", "get-url", "origin",
        )

        if remote.rstrip("/").removesuffix(".git") != (
            repository_url.removesuffix(".git")
        ):
            raise RuntimeError("This folder points to a different repository.")

        changes = run_git(
            "-C", str(repository_dir),
            "status", "--porcelain",
        )

        if changes:
            raise RuntimeError(
                "The repository contains local changes. "
                "They have been left untouched. Keep normalized outputs "
                "outside cvelistV5-live."
            )

        print("Downloading changes since the previous update...")

        output = run_git(
            "-C", str(repository_dir),
            "pull", "--ff-only",
        )

        print(output)

    data_dir = repository_dir / "cves"

    if not data_dir.is_dir():
        raise RuntimeError("The expected cves folder was not found.")

    print(f"\nCVE List is ready: {data_dir}")
    return data_dir

In [ ]:
cve_data_folder = update_cvelist()

nvd

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
import gzip
import hashlib
import os
import tempfile
import time


NVD_FEED_URL = "https://nvd.nist.gov/feeds/json/cve/2.0"
HEADERS = {"User-Agent": "Capstone-NVD-Sync/1.0"}
CHUNK_SIZE = 1024 * 1024  # Process 1 MB at a time


def calculate_sha256(file_path):
    """Calculate a file's checksum without loading it into memory."""
    checksum = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(CHUNK_SIZE), b""):
            checksum.update(chunk)

    return checksum.hexdigest()


def fetch_metadata(year):
    """Read NVD's checksum and size for a yearly feed."""
    url = f"{NVD_FEED_URL}/nvdcve-2.0-{year}.meta"
    request = Request(url, headers=HEADERS)

    with urlopen(request, timeout=60) as response:
        text = response.read().decode("utf-8-sig")

    metadata = {}

    for line in text.splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            metadata[key.strip()] = value.strip()

    expected_hash = metadata.get("sha256", "").lower()

    if (
        len(expected_hash) != 64
        or any(character not in "0123456789abcdef"
               for character in expected_hash)
    ):
        raise ValueError(f"Invalid checksum in NVD metadata for {year}")

    return metadata


def update_year(year, folder, attempts=3):
    """
    Skip unchanged files.
    Download, decompress and verify changed files before replacing them.
    """
    destination = folder / f"{year}.json"

    local_hash = (
        calculate_sha256(destination)
        if destination.exists()
        else None
    )

    for attempt in range(1, attempts + 1):
        temporary_path = None

        try:
            # Refresh metadata on every attempt in case NVD changes mid-download.
            metadata = fetch_metadata(year)
            expected_hash = metadata["sha256"].lower()

            if local_hash == expected_hash:
                print(f"{year}: already up to date")
                return "unchanged"

            print(f"{year}: downloading updated data...")

            url = f"{NVD_FEED_URL}/nvdcve-2.0-{year}.json.gz"
            request = Request(url, headers=HEADERS)

            # Temporary file is on the same drive for safe atomic replacement.
            descriptor, temporary_name = tempfile.mkstemp(
                prefix=f".{year}_",
                suffix=".json.tmp",
                dir=folder,
            )
            temporary_path = Path(temporary_name)

            checksum = hashlib.sha256()
            total_bytes = 0

            with os.fdopen(descriptor, "wb") as output:
                with urlopen(request, timeout=180) as response:
                    with gzip.GzipFile(fileobj=response) as compressed:
                        while True:
                            chunk = compressed.read(CHUNK_SIZE)

                            if not chunk:
                                break

                            output.write(chunk)
                            checksum.update(chunk)
                            total_bytes += len(chunk)

                output.flush()
                os.fsync(output.fileno())

            if checksum.hexdigest() != expected_hash:
                raise ValueError("Downloaded file checksum does not match NVD")

            expected_size = metadata.get("size")
            if expected_size and total_bytes != int(expected_size):
                raise ValueError("Downloaded file size does not match NVD")

            # Old file remains untouched until the new file passes verification.
            os.replace(temporary_path, destination)
            temporary_path = None

            print(f"{year}: updated successfully")
            return "updated"

        except Exception as error:
            print(f"{year}: attempt {attempt}/{attempts} failed: {error}")

            if attempt == attempts:
                print(f"{year}: FAILED — existing file was kept")
                return "failed"

            time.sleep(5 * attempt)

        finally:
            # Remove only this attempt's incomplete temporary download.
            if temporary_path is not None:
                temporary_path.unlink(missing_ok=True)


def update_nvd_data(folder="nvd_cve_data"):
    """Check all yearly feeds from 2002 through the current year."""
    folder = Path(folder).resolve()

    if not folder.is_dir():
        raise FileNotFoundError(
            f"Folder not found: {folder}\n"
            "Run the notebook from the folder containing nvd_cve_data, "
            "or pass its full path."
        )

    print(f"Updating: {folder}\n")

    results = {}

    for year in range(2002, datetime.now(timezone.utc).year + 1):
        results[year] = update_year(year, folder)
        time.sleep(1)  # Keep requests sequential and polite

    updated = [year for year, status in results.items() if status == "updated"]
    unchanged = [year for year, status in results.items() if status == "unchanged"]
    failed = [year for year, status in results.items() if status == "failed"]

    print("\nUPDATE COMPLETE")
    print(f"Updated:   {len(updated)}")
    print(f"Unchanged: {len(unchanged)}")
    print(f"Failed:    {len(failed)}")

    if failed:
        print("Failed years:", failed)
        print("Run the cell again to retry. Existing files were preserved.")

    return results

In [ ]:
results = update_nvd_data("nvd_cve_data")

cisa kev

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone
import csv
import os
import tempfile


CISA_KEV_URL = (
    "https://www.cisa.gov/sites/default/files/csv/"
    "known_exploited_vulnerabilities.csv"
)

HEADERS = {
    "User-Agent": "IITGN-Capstone-Data-Updater/1.0"
}


def update_cisa_kev(folder="."):
    """
    Download the complete current CISA KEV catalog.

    Creates:
        cisa_kev-YYYY-MM-DD.csv

    If today's file already exists, it is safely replaced.
    Older dated files are left untouched.
    """

    folder = Path(folder).resolve()
    folder.mkdir(parents=True, exist_ok=True)

    request = Request(CISA_KEV_URL, headers=HEADERS)
    temporary_path = None

    try:
        print("Downloading the latest CISA KEV catalog...")

        with urlopen(request, timeout=180) as response:
            last_modified = response.headers.get("Last-Modified")

            if last_modified:
                file_date = parsedate_to_datetime(
                    last_modified
                ).date().isoformat()
            else:
                file_date = datetime.now(
                    timezone.utc
                ).date().isoformat()

            descriptor, temporary_name = tempfile.mkstemp(
                prefix=".cisa_kev_",
                suffix=".csv.tmp",
                dir=folder,
            )
            temporary_path = Path(temporary_name)

            with os.fdopen(descriptor, "wb") as output:
                while True:
                    chunk = response.read(1024 * 1024)

                    if not chunk:
                        break

                    output.write(chunk)

                output.flush()
                os.fsync(output.fileno())

        # Check that the downloaded file is a valid KEV CSV.
        with temporary_path.open(
            "r",
            encoding="utf-8-sig",
            newline="",
        ) as file:
            reader = csv.DictReader(file)

            required_columns = {
                "cveID",
                "vendorProject",
                "product",
                "vulnerabilityName",
                "dateAdded",
                "requiredAction",
                "dueDate",
            }

            actual_columns = set(reader.fieldnames or [])

            if not required_columns.issubset(actual_columns):
                raise ValueError(
                    "The downloaded file does not look like "
                    "a valid CISA KEV catalog."
                )

            record_count = sum(1 for row in reader if row.get("cveID"))

        if record_count == 0:
            raise ValueError("The downloaded KEV catalog is empty.")

        destination = folder / f"cisa_kev-{file_date}.csv"

        # Replace only after the new file has passed validation.
        os.replace(temporary_path, destination)
        temporary_path = None

        print("CISA KEV update complete.")
        print(f"Records: {record_count:,}")
        print(f"Saved as: {destination}")

        return destination

    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


cisa_kev_file = update_cisa_kev()

epss

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from datetime import datetime, timezone
import csv
import gzip
import os
import re
import tempfile


EPSS_URL = (
    "https://epss.empiricalsecurity.com/"
    "epss_scores-current.csv.gz"
)

HEADERS = {
    "User-Agent": "IITGN-Capstone-Data-Updater/1.0"
}


def update_epss(folder="."):
    """
    Download and decompress the complete current EPSS dataset.

    Creates:
        epss_scores-YYYY-MM-DD.csv

    If that day's file already exists, it is safely replaced.
    Older dated files are left untouched.
    """

    folder = Path(folder).resolve()
    folder.mkdir(parents=True, exist_ok=True)

    request = Request(EPSS_URL, headers=HEADERS)
    temporary_path = None

    try:
        print("Downloading the latest EPSS dataset...")

        descriptor, temporary_name = tempfile.mkstemp(
            prefix=".epss_scores_",
            suffix=".csv.tmp",
            dir=folder,
        )
        temporary_path = Path(temporary_name)

        with os.fdopen(descriptor, "wb") as output:
            with urlopen(request, timeout=300) as response:
                # EPSS is downloaded as GZIP and saved as normal CSV.
                with gzip.GzipFile(fileobj=response) as compressed:
                    while True:
                        chunk = compressed.read(1024 * 1024)

                        if not chunk:
                            break

                        output.write(chunk)

            output.flush()
            os.fsync(output.fileno())

        # Read metadata from the first EPSS line.
        with temporary_path.open(
            "r",
            encoding="utf-8-sig",
            newline="",
        ) as file:
            first_line = file.readline().strip()

            score_date_match = re.search(
                r"score_date:(\d{4}-\d{2}-\d{2})",
                first_line,
            )

            if score_date_match:
                file_date = score_date_match.group(1)
            else:
                file_date = datetime.now(
                    timezone.utc
                ).date().isoformat()

            # EPSS files normally begin with a metadata line.
            if not first_line.startswith("#"):
                file.seek(0)

            reader = csv.DictReader(file)

            required_columns = {"cve", "epss", "percentile"}
            actual_columns = set(reader.fieldnames or [])

            if not required_columns.issubset(actual_columns):
                raise ValueError(
                    "The downloaded file does not look like "
                    "a valid EPSS dataset."
                )

            record_count = sum(1 for row in reader if row.get("cve"))

        if record_count == 0:
            raise ValueError("The downloaded EPSS dataset is empty.")

        destination = folder / f"epss_scores-{file_date}.csv"

        # Replace only after the new file has passed validation.
        os.replace(temporary_path, destination)
        temporary_path = None

        print("EPSS update complete.")
        print(f"Records: {record_count:,}")
        print(f"Saved as: {destination}")

        return destination

    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


epss_file = update_epss()